# NinaPro -- Results Analysis

Loads all finished MLflow runs from the local `mlruns/` directory (no database needed)
and provides a full comparative analysis with pandas, seaborn and matplotlib.

In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

MLFLOW_EXPERIMENT = "ninapro_emg_samplesplit_multiscale_short_10"   # must match explore_emg.ipynb
MLRUNS_DIR        = "/ai_workspace/hwoehrle/mlruns"        # relative to this notebook

mlflow.set_tracking_uri(MLRUNS_DIR)
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if exp is None:
    raise RuntimeError(
        f"Experiment '{MLFLOW_EXPERIMENT}' not found in '{MLRUNS_DIR}'.\n"
        "Run at least one training run in explore_emg.ipynb first."
    )

all_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    #filter_string="attributes.status = 'FINISHED'",
    max_results=5000,
)
print(f"Found {len(all_runs)} finished runs in experiment '{MLFLOW_EXPERIMENT}'")

In [ ]:
# Build flat DataFrame from runs + per-epoch metrics
records = []
epoch_records = []

for run in all_runs:
    p = run.data.params
    m = run.data.metrics

    row = {
        "run_id":        run.info.run_id,
        "run_name":      run.info.run_name,
        "subject":       int(p.get("subject", -1)),
        "kernels":       p.get("kernels", ""),
        "window_size":   int(p.get("window_size", -1)),
        "num_stages":    int(p.get("num_stages", -1)),
        "n_params":      int(p.get("n_params", 0)),
        "epochs":        int(p.get("epochs", 0)),
        "lr":            float(p.get("lr", 0)),
        "weight_decay":  float(p.get("weight_decay", 0)),
        "num_classes":   int(p.get("num_classes", 0)),
        "train_windows": int(p.get("train_windows", 0)),
        "test_windows":  int(p.get("test_windows", 0)),
        "final_acc":     m.get("final_test_acc", float("nan")),
        "test_acc":      m.get("test_acc", float("nan")),
        "num_conv_blocks": int(p.get("num_conv_blocks", 0)),
        "pool_size":     int(p.get("pool_size", -1)),
        "model_type":    p.get("model_type", "multiscale"),
        "split_mode":    p.get("split_mode", "repetition"),
        "dropout":       float(p.get("dropout", 0.5)),
    }
    records.append(row)

    # Load per-epoch accuracy curves
    for metric_key in ("train_acc", "test_acc"):
        history = client.get_metric_history(run.info.run_id, metric_key)
        for point in history:
            epoch_records.append({
                "run_id":      run.info.run_id,
                "subject":     row["subject"],
                "kernels":     row["kernels"],
                "window_size": row["window_size"],
                "num_stages":  row["num_stages"],
                "model_type":  row["model_type"],
                "split_mode":  row["split_mode"],
                "metric":      metric_key,
                "epoch":       point.step,
                "value":       point.value,
            })

df_all   = pd.DataFrame(records).sort_values(
    ["split_mode", "model_type", "subject", "kernels", "window_size", "num_stages"]
).reset_index(drop=True)
df_epoch_all = pd.DataFrame(epoch_records)

print(f"Runs loaded : {len(df_all)}")
print(f"Epoch points: {len(df_epoch_all)}")
print(f"\nRuns per split_mode x model_type:")
print(df_all.groupby(["split_mode", "model_type"]).size().rename("count").to_string())
df_all.head()

In [ ]:
# - Filter -
# Change these two variables to focus the analysis below on a specific subset.
# Use None to include all values.
#
#   SPLIT_MODE  : "repetition" | "sample" | None
#   MODEL_TYPE  : "multiscale" | "conventional" | None
# -
SPLIT_MODE = "sample"   # <- change here
MODEL_TYPE = "multiscale"            # <- change here (None = both models)

mask = pd.Series(True, index=df_all.index)
if SPLIT_MODE is not None:
    mask &= df_all["split_mode"] == SPLIT_MODE
if MODEL_TYPE is not None:
    mask &= df_all["model_type"] == MODEL_TYPE

df       = df_all[mask].reset_index(drop=True)
df_epoch = df_epoch_all[df_epoch_all["run_id"].isin(df["run_id"])].reset_index(drop=True)

print(f"Active filter -- split_mode={SPLIT_MODE!r}  model_type={MODEL_TYPE!r}")
print(f"Runs after filter : {len(df)}")
print(df.groupby(["split_mode", "model_type"]).size().rename("count").to_string())

In [ ]:
# drop all were we have window_size == 512
#df = df[df["window_size"] != 512].reset_index(drop=True)

# combine the entries for num_conv_blocks == 0 and num_conv_blocks == 1, set num_conv_blocks to "1" for those entries
#df.loc[df["num_conv_blocks"].isin([0, 1]), "num_conv_blocks"] = 1

## PTQ-Quantization: FP32 vs. INT8

In [ ]:
import sys, copy, ast, yaml
sys.path.insert(0, "/ai_workspace/hwoehrle/repos/emg_analysis/emg_classifier")

import torch
import numpy as np
import mlflow.pytorch
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.ao.quantization.quantize_pt2e import convert_pt2e, prepare_pt2e
from torch.ao.quantization.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer, get_default_x86_inductor_quantization_config,
)
from emg_worker import NinaProDatasetByRepetition, load_and_preprocess

NINAPRO_DIR     = "/ai_workspace/hwoehrle/data/ninapro"
N_CALIB_BATCHES = 32


def _make_quantizer():
    q = X86InductorQuantizer()
    q.set_global(get_default_x86_inductor_quantization_config())
    return q


def _find_model_uri(mlruns_dir, exp_id, run_id, artifact_name="model"):
    """Find the correct path for a stored MLflow model.

    MLflow >= 2.18 stores models under {exp_id}/models/m-{uuid}/ instead of
    {run_id}/artifacts/{name}/. Reads meta.yaml from all model directories and
    searches for matching source_run_id + name. Falls back to the legacy path.
    """
    models_dir = Path(mlruns_dir) / exp_id / "models"
    if models_dir.exists():
        for model_dir in sorted(models_dir.iterdir()):
            meta_file = model_dir / "meta.yaml"
            if not meta_file.exists():
                continue
            meta = yaml.safe_load(meta_file.read_text())
            if (meta.get("source_run_id") == run_id
                    and meta.get("name") == artifact_name):
                return meta["artifact_location"]
    # Legacy (MLflow < 2.18)
    return str(Path(mlruns_dir) / exp_id / run_id / "artifacts" / artifact_name)


def _loaders_from_run(run, ninapro_dir=NINAPRO_DIR):
    """Reconstruct train/test DataLoaders from the stored MLflow run parameters."""
    p          = run.data.params
    db         = int(p.get("db", 2))
    subject    = int(p["subject"])
    window_size= int(p["window_size"])
    step       = int(p.get("window_step", 32))
    batch_size = int(p.get("batch_size", 64))
    fs         = int(p.get("fs", 2000))
    split_mode = p.get("split_mode", "repetition")
    train_reps = ast.literal_eval(p.get("train_reps", "[1,3,4,6]"))
    test_reps  = ast.literal_eval(p.get("test_reps",  "[2,5]"))

    if split_mode == "sample":
        tmp = load_and_preprocess(db, subject, ninapro_dir, fs=fs)
        lbl = tmp["restimulus"] if tmp["restimulus"].size else tmp["stimulus"]
        all_labels = sorted(int(c) for c in np.unique(lbl) if c > 0)
        lmap = {orig: new for new, orig in enumerate(all_labels)}
        full_ds = NinaProDatasetByRepetition(
            db, subject, reps=None,
            window_size=window_size, step=step,
            label_map=lmap, ninapro_dir=ninapro_dir, fs=fs,
        )
        train_idx, test_idx = train_test_split(
            range(len(full_ds)), test_size=0.3, random_state=42, shuffle=True
        )
        train_ds, test_ds = Subset(full_ds, train_idx), Subset(full_ds, test_idx)
    else:
        tmp = load_and_preprocess(db, subject, ninapro_dir, fs=fs)
        lbl  = tmp["restimulus"] if tmp["restimulus"].size else tmp["stimulus"]
        reps = tmp["repetition"].astype(int)
        mask = np.isin(reps, train_reps)
        all_labels = sorted(int(c) for c in np.unique(lbl[mask]) if c > 0)
        lmap = {orig: new for new, orig in enumerate(all_labels)}
        train_ds = NinaProDatasetByRepetition(
            db, subject, train_reps,
            window_size=window_size, step=step,
            label_map=lmap, ninapro_dir=ninapro_dir, fs=fs,
        )
        test_ds = NinaProDatasetByRepetition(
            db, subject, test_reps,
            window_size=window_size, step=step,
            label_map=lmap, ninapro_dir=ninapro_dir, fs=fs,
            channel_stats=train_ds.channel_stats,
        )

    kw = dict(batch_size=batch_size, num_workers=0, pin_memory=False)
    return (DataLoader(train_ds, shuffle=False, **kw),
            DataLoader(test_ds,  shuffle=False, **kw))


def _eval_cpu(model, loader):
    try:
        model.eval()
    except NotImplementedError:
        torch.ao.quantization.move_exported_model_to_eval(model)
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x).argmax(1))
            trues.append(y)
    return accuracy_score(torch.cat(trues).numpy(), torch.cat(preds).numpy())


def quantize_run(run, artifact="model", n_calib=N_CALIB_BATCHES,
                 ninapro_dir=NINAPRO_DIR):
    """Load FP32 model from MLflow, apply PTQ, return fp32_acc/int8_acc/acc_delta.

    artifact: 'model' (final model) or e.g. 'model_epoch10' (checkpoint).
    """
    p      = run.data.params
    exp_id = run.info.experiment_id
    run_id = run.info.run_id

    uri = _find_model_uri(MLRUNS_DIR, exp_id, run_id, artifact)
    print(f"  Loading: {uri}")
    fp32 = mlflow.pytorch.load_model(uri, map_location="cpu").eval()

    train_loader, test_loader = _loaders_from_run(run, ninapro_dir)
    fp32_acc = _eval_cpu(fp32, test_loader)

    example  = (next(iter(train_loader))[0],)
    m        = copy.deepcopy(fp32).eval()
    exported = torch.export.export(m, example)
    prepared = prepare_pt2e(exported.module(), _make_quantizer())
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            prepared(x)
            if i + 1 >= n_calib:
                break
    int8     = convert_pt2e(prepared)
    int8_acc = _eval_cpu(int8, test_loader)

    return {
        "run_id":      run_id,
        "artifact":    artifact,
        "subject":     int(p["subject"]),
        "kernels":     p.get("kernels", ""),
        "window_size": int(p["window_size"]),
        "pool_size":   int(p.get("pool_size", 2)),
        "split_mode":  p.get("split_mode", "repetition"),
        "fp32_acc":    fp32_acc,
        "int8_acc":    int8_acc,
        "acc_delta":   int8_acc - fp32_acc,
    }

print("Helper functions loaded.")


In [ ]:
# - Single run: select hyperparameter combination -
SUBJECT     = 31
KERNELS     = "(3, 11, 23)"   # as stored by MLflow
WINDOW_SIZE = 512
POOL_SIZE   = 8

row = df[
    (df["subject"]     == SUBJECT) &
    (df["kernels"]     == KERNELS) &
    (df["window_size"] == WINDOW_SIZE) &
    (df["pool_size"]   == POOL_SIZE)
]
if row.empty:
    raise RuntimeError("No matching run found -- adjust filters.")

run = client.get_run(row.iloc[0]["run_id"])
print(f"Run: {run.info.run_name}")

# Quantize final model
r = quantize_run(run, artifact="model")
print(f"\n  FP32 : {r['fp32_acc']:.4f}")
print(f"  INT8 : {r['int8_acc']:.4f}")
print(f"  delta    : {r['acc_delta']:+.4f}")

# Optional: checkpoint after epoch 10 (only if trained with --checkpoint-epochs 10)
# r10 = quantize_run(run, artifact="model_epoch10")
# print(f"\n  Epoch-10 checkpoint FP32={r10['fp32_acc']:.4f}  INT8={r10['int8_acc']:.4f}")


In [ ]:
# - Batch over all runs in the active filter -
# df contains the already-filtered runs (cell "Filter" above).
# For a specific subject subset: df[df["subject"] == 16]

quant_results = []
for _, row in df.iterrows():
    run = client.get_run(row["run_id"])
    try:
        r = quantize_run(run)
        quant_results.append(r)
        print(f"s{r['subject']:>2}  k={r['kernels']:<18}  w={r['window_size']}  "
              f"FP32={r['fp32_acc']:.4f}  INT8={r['int8_acc']:.4f}  delta={r['acc_delta']:+.4f}")
    except Exception as e:
        print(f"ERROR  run={row['run_id'][:8]}  {e}")

df_quant = pd.DataFrame(quant_results)

# - Summary -
summary = (
    df_quant.groupby(["kernels", "window_size", "pool_size"])[["fp32_acc", "int8_acc", "acc_delta"]]
    .agg(["mean", "std"])
    .round(4)
)
display(summary)

# - Visualisation -
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy drop due to quantization
ax = axes[0]
order = df_quant.groupby("kernels")["acc_delta"].mean().sort_values().index
sns.boxplot(data=df_quant, x="kernels", y="acc_delta", order=order, ax=ax, palette="Reds_r")
sns.stripplot(data=df_quant, x="kernels", y="acc_delta", order=order, ax=ax,
              color="black", alpha=0.5, size=4)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_title("Accuracy drop from PTQ  (delta = INT8 − FP32)")
ax.set_xlabel("Kernel-Config"); ax.set_ylabel("delta Accuracy")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.tick_params(axis="x", rotation=20)

# FP32 vs INT8 directly compared
ax = axes[1]
df_melt = df_quant.melt(id_vars=["subject", "kernels"],
                         value_vars=["fp32_acc", "int8_acc"],
                         var_name="Typ", value_name="Accuracy")
order = df_quant.groupby("kernels")["fp32_acc"].mean().sort_values(ascending=False).index
sns.boxplot(data=df_melt, x="kernels", y="Accuracy", hue="Typ",
            order=order, ax=ax, palette={"fp32_acc": "#1976D2", "int8_acc": "#F57C00"})
ax.set_title("FP32 vs. INT8 Accuracy")
ax.set_xlabel("Kernel-Config")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.tick_params(axis="x", rotation=20)
ax.legend(title="", labels=["FP32", "INT8"])

plt.tight_layout()
plt.show()
